In [1]:
import pandas as pd
import numpy as np
df = pd.read_parquet("../data/model_sample.parquet")
train = df[df["sample"] == "train"].copy()

In [2]:
list(df.columns)

['id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 'dti_joint',
 'verificat

In [4]:
leakage_cols = [
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "out_prncp", "out_prncp_inv", "recoveries", "collection_recovery_fee",
    "last_fico_range_high", "last_fico_range_low", "last_credit_pull_d",
    "pymnt_plan", "hardship_flag", "hardship_type", "hardship_reason",
    "hardship_status", "deferral_term", "hardship_amount", "hardship_start_date",
    "hardship_end_date", "payment_plan_start_date", "hardship_length",
    "hardship_dpd", "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount", "hardship_last_payment_amount",
    "debt_settlement_flag",
]

lc_pricing_cols = ["grade", "sub_grade", "int_rate", "installment"]

id_text_cols = [
    "id", "url", "title", "emp_title", "policy_code",
    "funded_amnt", "funded_amnt_inv", "initial_list_status",
]

fair_lending_cols = ["addr_state", "zip_code"]

joint_cols = [
    "application_type", "annual_inc_joint", "dti_joint",
    "verification_status_joint", "revol_bal_joint",
    "sec_app_fico_range_low", "sec_app_fico_range_high",
    "sec_app_earliest_cr_line", "sec_app_inq_last_6mths", "sec_app_mort_acc",
    "sec_app_open_acc", "sec_app_revol_util", "sec_app_open_act_il",
    "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
]

late_added_cols = [
    "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m",
    "mths_since_rcnt_il", "total_bal_il", "il_util", "open_rv_12m",
    "open_rv_24m", "max_bal_bc", "all_util", "inq_fi", "total_cu_tl",
    "inq_last_12m",
]

meta_cols = ["loan_status", "issue_d", "term", "default", "sample",
             "quarter", "unresolved", "year"]

In [5]:
drop_list = (leakage_cols + lc_pricing_cols + id_text_cols + fair_lending_cols
             + joint_cols + late_added_cols + meta_cols)

candidate_features = [c for c in df.columns if c not in drop_list]
len(candidate_features)

62

In [6]:
train[late_added_cols].isna().groupby(train["year"]).mean().round(2).T

year,2012,2013,2014,2015
open_acc_6m,1.0,1.0,1.0,0.95
open_act_il,1.0,1.0,1.0,0.95
open_il_12m,1.0,1.0,1.0,0.95
open_il_24m,1.0,1.0,1.0,0.95
mths_since_rcnt_il,1.0,1.0,1.0,0.95
total_bal_il,1.0,1.0,1.0,0.95
il_util,1.0,1.0,1.0,0.95
open_rv_12m,1.0,1.0,1.0,0.95
open_rv_24m,1.0,1.0,1.0,0.95
max_bal_bc,1.0,1.0,1.0,0.95


In [7]:
pd.set_option("display.max_rows", 100)
train[candidate_features].isna().groupby(train["year"]).mean().round(2).T

year,2012,2013,2014,2015
loan_amnt,0.00,0.00,0.00,0.00
emp_length,0.04,0.05,0.06,0.07
home_ownership,0.00,0.00,0.00,0.00
annual_inc,0.00,0.00,0.00,0.00
verification_status,0.00,0.00,0.00,0.00
purpose,0.00,0.00,0.00,0.00
dti,0.00,0.00,0.00,0.00
delinq_2yrs,0.00,0.00,0.00,0.00
earliest_cr_line,0.00,0.00,0.00,0.00
fico_range_low,0.00,0.00,0.00,0.00


Finding: ~35 bureau fields are ~52% missing in 2012 and fully populated from 2013 — LC began reporting them mid-2012. Rather than treat “not yet collected” as a borrower signal, the training window starts in 2013. Train = 2013–2015 (~546k loans).